In [43]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error

In [44]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [45]:
df = pd.read_csv('../data/processed/dataset_long.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df.head()

,datetime,underlying_price,contract,iv,option_type,strike,hour,minute,day_of_week,date,session_progress,days_to_expiry,moneyness,log_moneyness,dist_from_atm,dist_from_atm_pct,is_ce,strike_rank,iv_neighbor_-2,iv_neighbor_-1,iv_neighbor_+1,iv_neighbor_+2,iv_neighbor_mean,wide_iv_neighbor_mean,iv_lag_1,iv_roll_mean_5,iv_roll_std_5,iv_roll_mean_10,mean_ce_iv,mean_pe_iv
0,2026-01-07 09:15:00,26111.65,NIFTY27JAN2625200CE,0.12662,CE,25200,9,15,2,2026-01-07,4860.0,20.2569,0.965086,-0.035538,911.65,0.034914,1,0.0,NaN,NaN,0.12330,0.11741,0.12330,0.11741,NaN,NaN,NaN,NaN,0.102447,0.102447
1,2026-01-07 09:20:00,26141.40,NIFTY27JAN2625200CE,0.08632,CE,25200,9,20,2,2026-01-07,4865.0,20.2535,0.963988,-0.036676,941.40,0.036012,1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.12662,NaN,NaN,NaN,0.098723,0.098723
2,2026-01-07 09:25:00,26139.35,NIFTY27JAN2625200CE,0.09147,CE,25200,9,25,2,2026-01-07,4870.0,20.2500,0.964064,-0.036598,939.35,0.035936,1,0.0,NaN,NaN,NaN,0.09514,NaN,0.09514,0.08632,0.106470,0.028496,NaN,0.091074,0.091074
3,2026-01-07 09:30:00,26128.95,NIFTY27JAN2625200CE,0.10860,CE,25200,9,30,2,2026-01-07,4875.0,20.2465,0.964447,-0.036200,928.95,0.035553,1,0.0,NaN,NaN,0.10842,0.11150,0.10842,0.11150,0.09147,0.101470,0.021932,0.101470,0.103252,0.103252
4,2026-01-07 09:35:00,26131.90,NIFTY27JAN2625200CE,0.10462,CE,25200,9,35,2,2026-01-07,4880.0,20.2431,0.964339,-0.036313,931.90,0.035661,1,0.0,NaN,NaN,0.10538,0.12459,0.10538,0.12459,0.10860,0.103252,0.018259,0.103252,0.104097,0.104097


In [46]:
df.columns

Index(['datetime', 'underlying_price', 'contract', 'iv', 'option_type',
       'strike', 'hour', 'minute', 'day_of_week', 'date', 'session_progress',
       'days_to_expiry', 'moneyness', 'log_moneyness', 'dist_from_atm',
       'dist_from_atm_pct', 'is_ce', 'strike_rank', 'iv_neighbor_-2',
       'iv_neighbor_-1', 'iv_neighbor_+1', 'iv_neighbor_+2',
       'iv_neighbor_mean', 'wide_iv_neighbor_mean', 'iv_lag_1',
       'iv_roll_mean_5', 'iv_roll_std_5', 'iv_roll_mean_10', 'mean_ce_iv',
       'mean_pe_iv'],
      dtype='str')

In [52]:
trading_days = sorted(df['datetime'].dt.date.unique())

folds = [
    {'name': 'fold1', 'train_dates': dates[:8],  'val_dates': dates[8:10]},
    {'name': 'fold2', 'train_dates': dates[:10], 'val_dates': dates[10:12]},
]

In [48]:
def make_holdout(val_df,random_state=2):
    np.random.seed(42)

    available_rows = val_df[val_df['iv'].notna()].index

    hidden_rows = np.random.choice(available_rows,size=int(0.2*len(available_rows)),replace=False)

    ground_truth = val_df.loc[hidden_rows,'iv'].values

    val_masked = val_df.copy()

    val_masked.loc[hidden_rows,'iv'] = np.nan

    return val_masked,hidden_rows,ground_truth

In [49]:
# evaluation functions for ml models
def eval_pred(actual,predicted):
    valid = ~np.isnan(predicted)

    mse = mean_squared_error(actual[valid],predicted[valid])
    coverage = valid.mean()

    print(f"MSE: {mse:6f}")
    print(f"Coverage: {coverage:6f}")

    return mse,coverage

In [64]:
import xgboost
import lightgbm
import catboost

### Model 1: XGBoost will all features

In [54]:
from xgboost import XGBRegressor

fold_mses, fold_coverages, feature_importances = [],[],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df[df['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df[df['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)
    feature_importances.append(xgb_model.feature_importances_)

print(f"xgb_mse: {np.mean(fold_mses):6f}")
print(f"xgb_coverage: {np.mean(fold_coverages):6f}")
print(f"feature importances: {feature_importances}")


MSE: 0.000114
Coverage: 1.000000
MSE: 0.000047
Coverage: 1.000000
xgb_mse: 0.000081
xgb_coverage: 1.000000
feature importances: [array([0.0042983 , 0.01186388, 0.00338471, 0.00345063, 0.00715866,
       0.00499231, 0.00423238, 0.0098574 , 0.00816558, 0.00841981,
       0.01185304, 0.        , 0.01034581, 0.00874176, 0.01494427,
       0.02264537, 0.02140105, 0.4981133 , 0.05849594, 0.00520056,
       0.22881566, 0.00379396, 0.02297753, 0.01251369, 0.01433442],
      dtype=float32), array([0.00247129, 0.00621403, 0.00108871, 0.00110867, 0.00310131,
       0.00215398, 0.00219415, 0.02390117, 0.00368832, 0.0064261 ,
       0.00643909, 0.0005458 , 0.00393209, 0.00472965, 0.00666619,
       0.0132345 , 0.00542649, 0.26722205, 0.03958256, 0.00331858,
       0.49992165, 0.00180436, 0.07908553, 0.00519456, 0.01054917],
      dtype=float32)]


In [59]:
avg_importance = np.mean(feature_importances,axis=0)

importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': avg_importance
})

importance_df = importance_df.sort_values(by='importance',ascending=False)

importance_df

,feature,importance
17,iv_neighbor_mean,0.382668
20,iv_roll_mean_5,0.364369
22,iv_roll_mean_10,0.051032
18,wide_iv_neighbor_mean,0.049039
15,iv_neighbor_+1,0.017940
7,moneyness,0.016879
16,iv_neighbor_+2,0.013414
24,mean_pe_iv,0.012442
14,iv_neighbor_-1,0.010805
10,dist_from_atm_pct,0.009146


In [60]:
selected_features = importance_df[importance_df['importance']>=0.005]['feature'].tolist()

selected_features

['iv_neighbor_mean',
 'iv_roll_mean_5',
 'iv_roll_mean_10',
 'wide_iv_neighbor_mean',
 'iv_neighbor_+1',
 'moneyness',
 'iv_neighbor_+2',
 'mean_pe_iv',
 'iv_neighbor_-1',
 'dist_from_atm_pct',
 'strike',
 'mean_ce_iv',
 'dist_from_atm',
 'strike_rank',
 'iv_neighbor_-2',
 'log_moneyness',
 'day_of_week']

In [61]:
df_pruned = df[selected_features+['iv','contract','option_type','date','datetime']].copy()
print(df_pruned.shape)
df_pruned.head()

(27300, 22)


,iv_neighbor_mean,iv_roll_mean_5,iv_roll_mean_10,wide_iv_neighbor_mean,iv_neighbor_+1,moneyness,iv_neighbor_+2,mean_pe_iv,iv_neighbor_-1,dist_from_atm_pct,strike,mean_ce_iv,dist_from_atm,strike_rank,iv_neighbor_-2,log_moneyness,day_of_week,iv,contract,option_type,date,datetime
0,0.12330,NaN,NaN,0.11741,0.12330,0.965086,0.11741,0.102447,NaN,0.034914,25200,0.102447,911.65,0.0,NaN,-0.035538,2,0.12662,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:15:00
1,NaN,NaN,NaN,NaN,NaN,0.963988,NaN,0.098723,NaN,0.036012,25200,0.098723,941.40,0.0,NaN,-0.036676,2,0.08632,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:20:00
2,NaN,0.106470,NaN,0.09514,NaN,0.964064,0.09514,0.091074,NaN,0.035936,25200,0.091074,939.35,0.0,NaN,-0.036598,2,0.09147,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:25:00
3,0.10842,0.101470,0.101470,0.11150,0.10842,0.964447,0.11150,0.103252,NaN,0.035553,25200,0.103252,928.95,0.0,NaN,-0.036200,2,0.10860,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:30:00
4,0.10538,0.103252,0.103252,0.12459,0.10538,0.964339,0.12459,0.104097,NaN,0.035661,25200,0.104097,931.90,0.0,NaN,-0.036313,2,0.10462,NIFTY27JAN2625200CE,CE,2026-01-07,2026-01-07 09:35:00


In [62]:
df_pruned.to_csv('../data/processed/dataset_long_pruned.csv',index=False)

### Model 2: XGBoost with pruned features

In [63]:
fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    xgb_model = XGBRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    xgb_model.fit(X_train,y_train)
    xgb_pred = xgb_model.predict(X_val)

    xgb_mse, xgb_coverage = eval_pred(ground_truth,xgb_pred)

    fold_mses.append(xgb_mse)
    fold_coverages.append(xgb_coverage)

print(f"xgbp_mse: {np.mean(fold_mses):6f}")
print(f"xgbp_coverage: {np.mean(fold_coverages):6f}")

MSE: 0.000104
Coverage: 1.000000
MSE: 0.000033
Coverage: 1.000000
xgbp_mse: 0.000069
xgbp_coverage: 1.000000


### Model 3: LightGBM

In [71]:
from lightgbm import LGBMRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    lgbm_model = LGBMRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        colsample_bytree= 0.8,
        random_state= 2,
    )

    lgbm_model.fit(X_train,y_train)
    lgbm_pred = lgbm_model.predict(X_val)

    lgbm_mse, lgbm_coverage = eval_pred(ground_truth,lgbm_pred)

    fold_mses.append(lgbm_mse)
    fold_coverages.append(lgbm_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"lgbm_mse: {np.mean(fold_mses):6f}")
print(f"lgbm_coverage: {np.mean(fold_coverages):6f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001111 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3598
[LightGBM] [Info] Number of data points in the train set: 13496, number of used features: 17
[LightGBM] [Info] Start training from score 0.106715
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain,

### Model 4: CatBoost

In [75]:
from catboost import CatBoostRegressor

fold_mses, fold_coverages = [],[]

drop_cols = ['iv','datetime','contract','option_type','date']

for fold in folds:
    train_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['train_dates'])].copy()

    val_df = df_pruned[df_pruned['datetime'].dt.date.isin(fold['val_dates'])].copy()

    val_masked, hidden_rows, ground_truth = make_holdout(val_df)

    # train data
    train_obs = train_df[train_df['iv'].notna()].copy()

    X_train = train_obs.drop(columns=drop_cols)
    y_train = train_obs['iv']

    # validation data
    X_val = val_masked.loc[hidden_rows].drop(columns=drop_cols)

    cb_model = CatBoostRegressor(
        n_estimators= 500,
        learning_rate= 0.05,
        max_depth= 6,
        subsample= 0.8,
        random_state= 2,
    )

    cb_model.fit(X_train,y_train)
    cb_pred = cb_model.predict(X_val)

    cb_mse, cb_coverage = eval_pred(ground_truth,cb_pred)

    fold_mses.append(cb_mse)
    fold_coverages.append(cb_coverage)

print(f"MSE from both folds: {fold_mses}")
print(f"cb_mse: {np.mean(fold_mses):6f}")
print(f"cb_coverage: {np.mean(fold_coverages):6f}")

0:	learn: 0.0120786	total: 164ms	remaining: 1m 21s
1:	learn: 0.0116134	total: 174ms	remaining: 43.2s
2:	learn: 0.0111750	total: 182ms	remaining: 30.1s
3:	learn: 0.0107577	total: 191ms	remaining: 23.7s
4:	learn: 0.0103597	total: 198ms	remaining: 19.6s
5:	learn: 0.0099842	total: 205ms	remaining: 16.9s
6:	learn: 0.0096301	total: 211ms	remaining: 14.9s
7:	learn: 0.0092875	total: 218ms	remaining: 13.4s
8:	learn: 0.0089566	total: 227ms	remaining: 12.4s
9:	learn: 0.0086560	total: 234ms	remaining: 11.5s
10:	learn: 0.0083585	total: 242ms	remaining: 10.8s
11:	learn: 0.0080878	total: 249ms	remaining: 10.1s
12:	learn: 0.0078296	total: 254ms	remaining: 9.53s
13:	learn: 0.0075801	total: 260ms	remaining: 9.04s
14:	learn: 0.0073521	total: 265ms	remaining: 8.58s
15:	learn: 0.0071296	total: 271ms	remaining: 8.21s
16:	learn: 0.0069189	total: 276ms	remaining: 7.85s
17:	learn: 0.0067197	total: 282ms	remaining: 7.55s
18:	learn: 0.0065286	total: 286ms	remaining: 7.25s
19:	learn: 0.0063502	total: 291ms	remain